# UTS EduBot — Chatbot Edukasi Sistem Informasi

Notebook Google Colab ini adalah panduan bertahap untuk membuat prototipe **Text → Qwen → Text**. Jalankan sel dari atas ke bawah. Proyek: chatbot edukasi Sistem Informasi, dikerjakan solo, dengan Python tingkat dasar.

> **Penting:** contoh dataset dan hasil di notebook ini hanya titik awal. Verifikasi setiap jawaban dengan buku/dosen/sumber tepercaya, lalu perluas datanya. Jangan menyimpulkan fine-tuning pasti membuat model lebih baik sebelum melihat evaluasi Anda sendiri.

Target notebook: setup → data → baseline Qwen → split → LoRA → inferensi → evaluasi → Gradio → checklist demo.

## 0. Sebelum mulai

Di Colab pilih **Runtime > Change runtime type > GPU** (T4/L4/A100 bila tersedia). Qwen 0.5B relatif ringan, tetapi GPU/VRAM dan kuota Colab tetap dapat membatasi proses. Jika GPU tidak tersedia, baseline mungkin masih berjalan lambat di CPU; fine-tuning sebaiknya ditunda.

In [ ]:
# Sel 1 — cek perangkat dan buat folder kerja
import os, json, random, gc, platform
from pathlib import Path

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('GPU tersedia:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
    else:
        print('Aktifkan GPU melalui Runtime > Change runtime type > GPU sebelum fine-tuning.')
except Exception as e:
    print('Pemeriksaan PyTorch gagal:', e)

PROJECT_DIR = Path('/content/edubot_uts')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('Folder proyek:', PROJECT_DIR)

In [ ]:
# Sel 2 — instal dependensi. Jalankan sekali; restart runtime jika Colab memintanya.
# Versi minimum dipakai untuk mengurangi masalah kompatibilitas antarlibrary.
%pip -q install -U 'transformers>=4.45,<5' 'accelerate>=0.34' 'peft>=0.13' 'datasets>=3.0' 'bitsandbytes>=0.43' 'gradio>=5.0' sentencepiece

import transformers, peft, datasets, accelerate
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)
print('datasets:', datasets.__version__)
print('accelerate:', accelerate.__version__)
print('Catatan: jika impor atau instalasi gagal, pilih Runtime > Restart session lalu ulangi dari Sel 1.')

## 1. Dataset contoh

Format setiap contoh adalah `system`, `user`, dan `assistant`. Jawaban contoh dibuat singkat agar mudah diaudit. Tambahkan variasi pertanyaan, istilah kuliah Anda, dan rujukan materi. Hindari memasukkan data pribadi mahasiswa atau jawaban yang belum terverifikasi. Untuk tugas yang lebih kuat, targetkan data jauh lebih banyak dan beragam daripada 20 contoh ini.

In [ ]:
# Sel 3 — buat dataset Q&A contoh dan simpan sebagai JSONL
SYSTEM_PROMPT = ('Anda adalah EduBot, asisten edukasi Sistem Informasi. Jawab dalam bahasa Indonesia, '
                 'ringkas, akurat, dan jujur. Jika konteks tidak cukup, katakan bahwa pengguna perlu memeriksa materi kuliah atau dosen.')
qa_examples = [
    ('Apa itu Sistem Informasi?', 'Sistem Informasi adalah gabungan manusia, proses, data, perangkat keras, perangkat lunak, dan jaringan untuk mengumpulkan, mengolah, menyimpan, serta menyebarkan informasi guna mendukung organisasi.'),
    ('Apa perbedaan data dan informasi?', 'Data adalah fakta mentah, misalnya angka penjualan. Informasi adalah data yang sudah diolah sehingga bermakna untuk pengambilan keputusan.'),
    ('Sebutkan komponen utama Sistem Informasi.', 'Komponen umum Sistem Informasi meliputi manusia, prosedur atau proses, data, perangkat keras, perangkat lunak, dan jaringan atau komunikasi.'),
    ('Apa fungsi database?', 'Database menyimpan data secara terstruktur agar data dapat dicari, diperbarui, dijaga konsistensinya, dan digunakan bersama sesuai hak akses.'),
    ('Apa itu DBMS?', 'DBMS atau Database Management System adalah perangkat lunak untuk membuat, mengelola, mengakses, dan mengamankan database, misalnya MySQL atau PostgreSQL.'),
    ('Apa perbedaan primary key dan foreign key?', 'Primary key mengidentifikasi setiap baris secara unik dalam tabel. Foreign key menghubungkan tabel dengan merujuk primary key pada tabel lain.'),
    ('Apa itu SDLC?', 'SDLC adalah siklus hidup pengembangan sistem. Tahap umumnya mencakup perencanaan, analisis kebutuhan, perancangan, implementasi, pengujian, penerapan, dan pemeliharaan.'),
    ('Mengapa analisis kebutuhan penting?', 'Analisis kebutuhan membantu memastikan sistem dibangun untuk masalah dan kebutuhan pengguna yang benar, sehingga risiko salah fitur dan perubahan mahal dapat dikurangi.'),
    ('Apa itu UML?', 'UML adalah bahasa pemodelan visual untuk menjelaskan rancangan sistem. Contohnya use case diagram, class diagram, dan sequence diagram.'),
    ('Apa tujuan use case diagram?', 'Use case diagram menggambarkan aktor dan layanan atau fungsi sistem dari sudut pandang pengguna, sehingga batas sistem lebih mudah dipahami.'),
    ('Apa itu normalisasi database?', 'Normalisasi adalah pengorganisasian tabel untuk mengurangi redundansi dan anomali data. Penerapannya perlu disesuaikan dengan kebutuhan sistem dan performa.'),
    ('Jelaskan contoh input, process, output.', 'Pada sistem nilai: input adalah nilai tugas dan ujian, proses adalah perhitungan berdasarkan bobot, dan output adalah nilai akhir atau laporan nilai.'),
    ('Apa itu ERP?', 'ERP atau Enterprise Resource Planning adalah sistem terintegrasi yang membantu mengelola proses organisasi seperti keuangan, persediaan, SDM, dan pengadaan.'),
    ('Apa manfaat sistem informasi bagi manajer?', 'Sistem informasi dapat menyediakan laporan dan analisis tepat waktu untuk membantu perencanaan, pengendalian, koordinasi, dan pengambilan keputusan.'),
    ('Apa risiko keamanan informasi?', 'Risiko meliputi akses tanpa izin, kebocoran data, malware, kehilangan data, dan kesalahan manusia. Pengendalian dapat berupa kata sandi kuat, hak akses, backup, dan pelatihan.'),
    ('Apa prinsip CIA dalam keamanan informasi?', 'CIA berarti Confidentiality, Integrity, Availability: menjaga kerahasiaan, keutuhan, dan ketersediaan informasi.'),
    ('Apa itu cloud computing?', 'Cloud computing adalah penyediaan sumber daya komputasi melalui internet sesuai kebutuhan, misalnya penyimpanan, server, atau aplikasi.'),
    ('Apa beda waterfall dan agile?', 'Waterfall cenderung berurutan per tahap, sedangkan agile iteratif dengan umpan balik berkala. Pemilihan pendekatan bergantung pada kebutuhan dan perubahan proyek.'),
    ('Apa itu dashboard?', 'Dashboard adalah tampilan ringkas indikator dan visualisasi penting untuk memantau kondisi atau kinerja dan membantu pengguna mengambil keputusan.'),
    ('Apa etika penggunaan AI di kampus?', 'Gunakan AI secara transparan, patuhi aturan akademik, jangan mengklaim karya AI sebagai karya sendiri, lindungi data pribadi, dan verifikasi keluaran AI sebelum digunakan.')
]
records = [{'system': SYSTEM_PROMPT, 'user': q, 'assistant': a} for q, a in qa_examples]
DATA_PATH = PROJECT_DIR / 'edubot_si_qa.jsonl'
with DATA_PATH.open('w', encoding='utf-8') as f:
    for row in records:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(f'Tersimpan {len(records)} contoh ke: {DATA_PATH}')
print(json.dumps(records[0], ensure_ascii=False, indent=2))
print('\nTUGAS ANDA: baca ulang setiap jawaban, koreksi bila perlu, lalu tambah contoh sebelum pelatihan final.')

## 2. Load Qwen pretrained dan baseline

Model utama adalah `Qwen/Qwen2.5-0.5B-Instruct`. Jika model/akses Hugging Face tidak tersedia, sel akan menampilkan pesan tindakan. Kuantisasi 4-bit menghemat VRAM, tetapi perlu GPU CUDA dan `bitsandbytes`; notebook otomatis mencoba mode tersebut lalu memberi petunjuk bila gagal.

In [ ]:
# Sel 4 — pemuatan model pretrained yang hemat memori
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
USE_4BIT = torch.cuda.is_available()  # ubah False bila 4-bit bermasalah

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {'low_cpu_mem_usage': True}
    if torch.cuda.is_available():
        load_kwargs['device_map'] = 'auto'
    if USE_4BIT:
        load_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
        )
    elif torch.cuda.is_available():
        load_kwargs['torch_dtype'] = torch.float16

    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
    base_model.config.pad_token_id = tokenizer.pad_token_id
    print('Model pretrained berhasil dimuat. 4-bit:', USE_4BIT)
except Exception as e:
    raise RuntimeError(f'Gagal memuat {MODEL_ID}. Cek internet/GPU/versi library. Detail: {e}')

def generate_answer(model, question, max_new_tokens=160):
    """Menghasilkan jawaban; gunakan pertanyaan edukasi dan periksa faktanya."""
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

contoh_pertanyaan = 'Jelaskan mengapa backup data penting bagi organisasi.'
print('Pertanyaan:', contoh_pertanyaan)
print('Jawaban baseline:', generate_answer(base_model, contoh_pertanyaan))

## 3. Split tanpa kebocoran dan preprocessing chat template

Data uji dipisahkan **sebelum** pelatihan dan tidak boleh ditambahkan ke data latih. Dengan dataset sangat kecil, hasil evaluasi sangat tidak stabil—ini latihan alur kerja, bukan bukti kualitas umum.

In [ ]:
# Sel 5 — muat JSONL, split tetap/reproducible, dan terapkan template Qwen
from datasets import Dataset

with DATA_PATH.open(encoding='utf-8') as f:
    loaded_records = [json.loads(line) for line in f if line.strip()]
raw_ds = Dataset.from_list(loaded_records)
split_ds = raw_ds.train_test_split(test_size=0.20, seed=42, shuffle=True)
train_raw, test_raw = split_ds['train'], split_ds['test']
print('Train:', len(train_raw), '| Test:', len(test_raw))
print('Contoh pertanyaan TEST (jangan masukkan ke train):', test_raw[0]['user'])

MAX_LENGTH = 512
def tokenize_supervised(example):
    # Prompt tanpa jawaban dipakai untuk menutupi token instruksi; loss hanya menghukum jawaban asisten.
    messages = [
        {'role': 'system', 'content': example['system']},
        {'role': 'user', 'content': example['user']},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    full_text = prompt_text + example['assistant'] + tokenizer.eos_token
    full = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)
    prompt_ids = tokenizer(prompt_text, truncation=True, max_length=MAX_LENGTH, add_special_tokens=False)['input_ids']
    labels = full['input_ids'].copy()
    labels[:min(len(prompt_ids), len(labels))] = [-100] * min(len(prompt_ids), len(labels))
    full['labels'] = labels
    return full

train_ds = train_raw.map(tokenize_supervised, remove_columns=train_raw.column_names)
test_ds = test_raw.map(tokenize_supervised, remove_columns=test_raw.column_names)
print('Token contoh:', len(train_ds[0]['input_ids']))
print('Token prompt yang ditutup dari loss:', sum(x == -100 for x in train_ds[0]['labels']))

In [ ]:
# Sel 6 — simpan jawaban baseline untuk PERTANYAAN TEST yang sama
# Ini dilakukan sebelum adaptasi. Jangan memakai jawaban referensi saat mengirim prompt ke model.
baseline_outputs = []
for row in test_raw:
    try:
        baseline_outputs.append({
            'question': row['user'], 'reference': row['assistant'],
            'baseline': generate_answer(base_model, row['user'])
        })
    except Exception as e:
        baseline_outputs.append({'question': row['user'], 'reference': row['assistant'], 'baseline': f'[ERROR] {e}'})
BASELINE_PATH = PROJECT_DIR / 'baseline_test_outputs.json'
BASELINE_PATH.write_text(json.dumps(baseline_outputs, ensure_ascii=False, indent=2), encoding='utf-8')
print('Baseline tersimpan:', BASELINE_PATH)
print(json.dumps(baseline_outputs[0], ensure_ascii=False, indent=2))

## 4. Fine-tuning hemat memori dengan PEFT LoRA

LoRA melatih adapter kecil, bukan semua parameter Qwen. Konfigurasi ini sengaja konservatif: batch 1, akumulasi gradien, satu epoch, panjang 512. Naikkan data/epoch hanya setelah alur berhasil. Jika kehabisan memori, turunkan `MAX_LENGTH` ke 256, gunakan 4-bit, atau hentikan aplikasi lain di Colab.

In [ ]:
# Sel 7 — siapkan LoRA, data collator, dan Trainer
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

if not torch.cuda.is_available():
    raise RuntimeError('Fine-tuning dibatalkan demi kestabilan: aktifkan GPU Colab, lalu ulangi dari Sel 1.')

if USE_4BIT:
    base_model = prepare_model_for_kbit_training(base_model)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()

# Nama layer Qwen yang tersedia dicek agar error versi/model lebih mudah dipahami.
candidate_targets = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
available_names = {name.split('.')[-1] for name, _ in base_model.named_modules()}
target_modules = [name for name in candidate_targets if name in available_names]
if not target_modules:
    raise RuntimeError('Layer target LoRA tidak ditemukan. Cetak nama modul model dan sesuaikan target_modules.')
print('Target LoRA:', target_modules)

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
                         task_type=TaskType.CAUSAL_LM, target_modules=target_modules)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

def causal_lm_collator(features):
    # Padding input dan labels dibuat sejajar; -100 diabaikan oleh loss.
    max_len = max(len(f['input_ids']) for f in features)
    batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
    for f in features:
        pad = max_len - len(f['input_ids'])
        batch['input_ids'].append(f['input_ids'] + [tokenizer.pad_token_id] * pad)
        batch['attention_mask'].append(f['attention_mask'] + [0] * pad)
        batch['labels'].append(f['labels'] + [-100] * pad)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

ADAPTER_DIR = PROJECT_DIR / 'qwen_edubot_lora'
# Transformers versi baru memakai nama eval_strategy (versi lama memakai evaluation_strategy).
training_args = TrainingArguments(
    output_dir=str(PROJECT_DIR / 'training_checkpoints'),
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, num_train_epochs=1, learning_rate=2e-4,
    logging_steps=1, save_strategy='no', eval_strategy='epoch',
    fp16=True, report_to='none', remove_unused_columns=False,
    optim='paged_adamw_8bit', seed=42
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds,
                  data_collator=causal_lm_collator)
print('Trainer siap. Jika muncul error argumen versi, upgrade library pada Sel 2 dan restart runtime.')

In [ ]:
# Sel 8 — jalankan adaptasi lalu simpan adapter LoRA (bukan seluruh model dasar)
try:
    train_result = trainer.train()
    trainer.save_model(str(ADAPTER_DIR))
    tokenizer.save_pretrained(str(ADAPTER_DIR))
    metrics = trainer.evaluate()
    (PROJECT_DIR / 'training_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    print('Adapter tersimpan di:', ADAPTER_DIR)
    print('Metrik loss:', metrics)
except torch.cuda.OutOfMemoryError:
    torch.cuda.empty_cache()
    raise RuntimeError('GPU kehabisan memori. Turunkan MAX_LENGTH ke 256, pertahankan batch=1, lalu restart runtime dan ulangi.')
except Exception as e:
    raise RuntimeError(f'Pelatihan gagal: {type(e).__name__}: {e}')

# Loss bukan satu-satunya ukuran kualitas chatbot; gunakan perbandingan jawaban pada Sel 10.

In [ ]:
# Sel 9 — inferensi setelah adaptasi. model sekarang berisi adapter LoRA aktif.
model.config.use_cache = True
pertanyaan_baru = 'Bagaimana cara mahasiswa menjaga keamanan akun kampus?'
print('Pertanyaan:', pertanyaan_baru)
print('Jawaban model adaptasi:', generate_answer(model, pertanyaan_baru))
print('\nCatatan: jawaban tetap perlu diverifikasi; adapter pada data kecil dapat menghafal atau menurunkan kemampuan umum.')

## 5. Evaluasi sebelum vs sesudah

Bandingkan jawaban baseline dan adaptasi pada **pertanyaan test yang sama**. Beri skor manual 0–2 untuk akurasi, relevansi, kejelasan, dan keamanan/kejujuran. Total maksimum 8. Jangan hanya memilih jawaban yang lebih panjang; cek kesesuaiannya dengan referensi dan materi kuliah.

In [ ]:
# Sel 10 — buat tabel perbandingan dan file rubric yang dapat diisi manual
import pandas as pd
comparison = []
for before in baseline_outputs:
    try:
        after = generate_answer(model, before['question'])
    except Exception as e:
        after = f'[ERROR] {e}'
    comparison.append({
        'question': before['question'], 'reference': before['reference'],
        'baseline': before['baseline'], 'adapted': after,
        'baseline_score_0_8': '', 'adapted_score_0_8': '', 'reviewer_notes': ''
    })
comparison_df = pd.DataFrame(comparison)
COMPARISON_CSV = PROJECT_DIR / 'comparison_rubric.csv'
COMPARISON_JSON = PROJECT_DIR / 'comparison_outputs.json'
comparison_df.to_csv(COMPARISON_CSV, index=False, encoding='utf-8-sig')
COMPARISON_JSON.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
print('Isi skor setelah membaca jawaban. File:', COMPARISON_CSV)
display(comparison_df[['question', 'reference', 'baseline', 'adapted']])

RUBRIK = {
    'akurasi': '0=salah/berbahaya, 1=sebagian benar, 2=benar dan dapat diverifikasi',
    'relevansi': '0=tidak menjawab, 1=sebagian, 2=langsung menjawab',
    'kejelasan': '0=sulit dipahami, 1=cukup, 2=jelas dan ringkas',
    'keamanan_kejujuran': '0=menyesatkan, 1=netral, 2=mengakui batasan bila perlu'
}
print(json.dumps(RUBRIK, ensure_ascii=False, indent=2))
print('Kesimpulan yang benar harus berdasarkan skor dan catatan Anda; hasil adaptasi bisa lebih baik, sama saja, atau lebih buruk.')

## 6. Prototype Gradio

Aplikasi berikut memakai model adaptasi yang masih tersimpan di memori. Jalankan setelah Sel 8. Pada Colab, Gradio akan menampilkan tautan publik sementara. Jangan membagikan contoh yang berisi data pribadi.

In [ ]:
# Sel 11 — antarmuka chatbot sederhana
import gradio as gr

def chat_edubot(message, history):
    if not message or not message.strip():
        return 'Silakan tulis pertanyaan tentang Sistem Informasi.'
    try:
        return generate_answer(model, message.strip())
    except Exception as e:
        return f'Terjadi kesalahan saat membuat jawaban: {type(e).__name__}. Periksa sel pemuatan model dan GPU.'

demo = gr.ChatInterface(
    fn=chat_edubot,
    title='EduBot Sistem Informasi (Prototype UTS)',
    description='Tanyakan konsep Sistem Informasi. Jawaban AI perlu diverifikasi dengan materi kuliah/dosen.',
    examples=['Apa beda data dan informasi?', 'Jelaskan fungsi primary key.', 'Apa manfaat dashboard?']
)
demo.launch(share=True, debug=False)  # hentikan sel ini bila sudah selesai demonstrasi

## 7. Dokumentasi, troubleshooting, dan checklist demo

**Troubleshooting cepat**
- `CUDA out of memory`: ubah `MAX_LENGTH=256`, tetap batch 1, gunakan GPU, restart runtime, dan ulangi dari awal.
- Error `bitsandbytes`/4-bit: pada Sel 4 ubah `USE_4BIT = False`; model akan butuh VRAM lebih besar.
- Error versi `Trainer` atau `PEFT`: ulangi Sel 2, restart runtime, lalu jalankan semua sel berurutan.
- Model tidak terunduh: periksa koneksi internet Colab atau coba lagi; jangan mengganti model tanpa mencatat perubahan.
- Jawaban buruk/halusinasi: periksa dan perluas dataset terverifikasi, gunakan prompt yang jelas, evaluasi ulang. Jangan menganggap LoRA menyelesaikannya otomatis.

**Checklist demonstrasi UTS**
- [ ] Masalah/domain: edukasi konsep Sistem Informasi; input teks, output teks.
- [ ] Jelaskan Qwen pretrained (`Qwen/Qwen2.5-0.5B-Instruct`) dan alasan pemilihan model kecil.
- [ ] Tunjukkan format serta proses verifikasi/penambahan dataset JSONL.
- [ ] Tunjukkan preprocessing chat template dan split train/test tanpa kebocoran.
- [ ] Jelaskan LoRA/PEFT, konfigurasi hemat memori, dan keterbatasan GPU Colab.
- [ ] Demo baseline dan model adaptasi pada pertanyaan test yang sama.
- [ ] Tampilkan `comparison_rubric.csv`, skor manual, dan kesimpulan yang jujur.
- [ ] Demo Gradio dengan pertanyaan baru dan sebutkan keterbatasan/risiko halusinasi.

Artefak penting tersimpan di `/content/edubot_uts/`: dataset JSONL, output baseline, adapter LoRA, metrik, dan tabel evaluasi. Unduh folder tersebut dari panel Files Colab atau hubungkan Google Drive bila diperlukan.

In [ ]:
# Sel 12 — ringkasan artefak untuk dokumentasi laporan
print('Isi folder proyek:')
for p in sorted(PROJECT_DIR.rglob('*')):
    if p.is_file():
        print('-', p.relative_to(PROJECT_DIR), f'({p.stat().st_size / 1024:.1f} KB)')

print('\nTemplate kesimpulan laporan (isi setelah eksperimen):')
print('Model baseline dan LoRA dibandingkan pada', len(test_raw), 'pertanyaan test yang tidak digunakan saat training.')
print('Berdasarkan rubric manual, [isi hasil Anda]. Keterbatasan: dataset kecil, evaluasi manual, dan jawaban tetap perlu verifikasi.')